# Alice/Bob без пошагового барьера
Qwen3-4B без thinking, 1024 токена на агента.
Независимые циклы; эксклюзивный доступ к кешу только при публикации.
Kernel: **AsyncLLM (uv)**.


In [1]:
import asyncio
import gc
import os
import re
import sys
import time
from contextlib import aclosing
from pathlib import Path

from IPython.display import Markdown, display

ROOT = Path.cwd().parents[1] if Path.cwd().name == "parallel_thinking" else Path.cwd()
os.chdir(ROOT)
for path in (ROOT / "python", ROOT / "scripts/async_thoughts", ROOT / "scripts/parallel_thinking"):
    sys.path.insert(0, str(path))
os.environ.update(
    CUDA_DEVICE_ORDER="PCI_BUS_ID",
    CUDA_VISIBLE_DEVICES="1",
    TVM_FFI_CUDA_ARCH_LIST="8.6",
    OMP_NUM_THREADS="8",
)
os.environ["PATH"] = f"{Path(sys.executable).parent}:/usr/local/cuda/bin:" + os.environ["PATH"]

import torch
from async_helpers import PublicationGate, isolated, one_token, recorded_example, together
from async_thoughts.training import DemoRecorder
from hogwild_formatting import MathFormatting
from minisgl.core import SamplingParams
from minisgl.llm import AsyncLLM
from minisgl.shared_cache import AsyncContext
from minisgl.training.replay import Replay

TEACHER = "models/Qwen3-4B"
WORKERS = ("Alice", "Bob")
PROBLEM = (
    "Three vertices of a cube are $P=(7,12,10)$ , $Q=(8,8,1)$ , and "
    "$R=(11,3,9)$ . What is the surface area of the cube?"
)

/home/uskovev/async_llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
async def solve_no_barrier(model, *, max_steps=1024, prompt_ids=None, bob_delay=0.0):
    started = time.perf_counter()
    llm = AsyncLLM(
        str(model),
        dtype=torch.bfloat16,
        memory_ratio=0.65,
        max_running_req=4,
        max_seq_len_override=16384,
        cuda_graph_bs=[1, 2],
        cuda_graph_max_bs=2,
        shared_cuda_graph_max_depth=6,
        shared_cuda_graph_prefill_rows=[32, 256],
        max_prefill_rows=256,
        distributed_addr="tcp://127.0.0.1:0",
    )
    llm = DemoRecorder(llm)
    blocks = []
    try:
        tokenizer = llm.tokenizer
        encode = lambda text: tokenizer.encode(text, add_special_tokens=False)

        fmt = MathFormatting(
            tokenizer, WORKERS, enable_thinking=False
        )
        for _ in range(6):
            blocks.append(await llm.create_block())
        common, others_header, own_header, alice, bob, answer_block = blocks
        worker_blocks = [alice, bob]
        contexts = [
            AsyncContext(cache_view=[common, others_header, bob, own_header, alice]),
            AsyncContext(cache_view=[common, others_header, alice, own_header, bob]),
        ]
        initial_ids = (
            list(prompt_ids) if prompt_ids is not None else encode(fmt.apply_chat_template(PROBLEM))
        )

        await llm.forward(initial_ids, write_to=common, return_logits=False)
        await together(
            llm.forward(
                encode(fmt.current_step_header), write_to=others_header, return_logits=False
            ),
            llm.forward(
                encode(fmt.current_worker_header), write_to=own_header, return_logits=False
            ),
        )

        prefixes = [
            f"{fmt.get_step_prefix(WORKERS[0], 1)}Hi, I'm {WORKERS[0]}. Here's how we can collaborate",
            f"{fmt.get_step_prefix(WORKERS[1], 1)}Hi, I'm {WORKERS[1]}.",
        ]
        current = [encode(prefix) for prefix in prefixes]

        await together(
            *(
                llm.forward(
                    ids[:-1],
                    cache_view=[common, others_header, own_header, block],
                    return_logits=False,
                )
                for ids, block in zip(current, worker_blocks)
            )
        )
        for ctx, ids in zip(contexts, current):
            ctx.next_input_id = ids[-1]

        history, events = [], []
        step_indices, generated = [1, 1], [0, 0]
        sampled = [[], []]
        forbidden = [
            tokenizer.vocab[t]
            for t in ("<|im_start|>", "<|endoftext|>")
            if t in tokenizer.vocab
        ]

        eos = tokenizer.eos_token_id
        forbidden = [token for token in forbidden if token != eos]
        generation_started = time.perf_counter()

        gate = PublicationGate()
        stopped = False
        progress = []

        async def generate_agent(i):
            nonlocal stopped
            ctx = contexts[i]
            for local_step in range(1, max_steps + 1):
                if stopped:
                    break

                if i == 1 and bob_delay and local_step % 32 == 0:
                    await asyncio.sleep(bob_delay)
                async with gate.forward():
                    token = await one_token(llm, ctx, forbidden)
                    current[i].append(token)
                    generated[i] += 1
                    sampled[i].append(token)
                    progress.append(tuple(generated))
                if token == eos:
                    stopped = True
                    break
                if fmt.is_end_of_step(current[i]):

                    async with gate.publish():
                        await llm.forward(cache_view=ctx, return_logits=False)
                        block = worker_blocks[i]
                        assert block.token_ids == current[i]
                        await llm.append_block(common, block)
                        history.extend(current[i])
                        assert common.token_ids == initial_ids + history
                        events.append(
                            {
                                "worker": WORKERS[i],
                                "step": step_indices[i],
                                "tokens": len(current[i]),
                                "generated": tuple(generated),
                                "text": tokenizer.decode(current[i]),
                            }
                        )
                        await llm.free_block(block)
                        step_indices[i] += 1
                        current[i] = encode(fmt.get_step_prefix(WORKERS[i], step_indices[i]))
                        await llm.forward(current[i][:-1], cache_view=ctx, return_logits=False)
                        ctx.next_input_id = current[i][-1]
                assert ctx.output_block.token_ids == current[i][:-1]
                assert ctx.next_input_id == current[i][-1]

        await together(generate_agent(0), generate_agent(1))
        iteration = max(generated)

        await together(*(llm.forward(cache_view=ctx, return_logits=False) for ctx in contexts))
        for i, ctx in enumerate(contexts):
            assert ctx.output_block.token_ids == current[i]
        parallel_seconds = time.perf_counter() - generation_started
        transcript = fmt.history_header + fmt.sep + tokenizer.decode(history)
        transcript += fmt.sep + fmt.current_step_header
        transcript += "".join(
            tokenizer.decode(ids) + fmt.incomplete_step + fmt.sep for ids in current
        )

        suffix = fmt.s1_finisher_suffix
        answer_ids = encode(suffix)
        answer_ctx = AsyncContext(
            cache_view=[common, others_header, alice, own_header, bob, answer_block]
        )
        await llm.forward(answer_ids[:-1], cache_view=answer_ctx, return_logits=False)
        answer_ctx.next_input_id = answer_ids[-1]
        answer_tokens = []
        async with aclosing(
            llm.async_generate(
                answer_ctx,
                max_steps=64,
                forbid_ids=forbidden,
                sampling_params=SamplingParams(temperature=0.0, max_tokens=64),
            )
        ) as stream:
            async for token in stream:
                if token == eos:
                    break
                answer_tokens.append(token)
                if fmt.get_final_answer(suffix + tokenizer.decode(answer_tokens)) is not None:
                    break
        answer_text = suffix + tokenizer.decode(answer_tokens)
        result = {
            "model": str(model),
            "problem": PROBLEM,
            "rounds": iteration,
            "generated_tokens": dict(zip(WORKERS, generated)),
            "completed_steps": len(events),
            "merge_events": events,
            "transcript": transcript,
            "answer_text": answer_text,
            "answer": fmt.get_final_answer(answer_text),
            "parallel_seconds": parallel_seconds,
            "total_seconds": time.perf_counter() - started,
        }
        result["generated_text"] = {w: tokenizer.decode(ids) for w, ids in zip(WORKERS, sampled)}
        result["prompt_ids"] = initial_ids
        result["history_ids"] = history
        result["current_ids"] = current
        result["answer_ids"] = answer_tokens
        result["progress"] = progress
        result["max_token_gap"] = max(abs(a - b) for a, b in progress)
        result["example"] = recorded_example(llm)
        return result
    finally:

        try:
            for block in blocks:
                await llm.free_block(block)
        finally:
            await llm.close()
            gc.collect()
            torch.cuda.empty_cache()

def show_result(result):
    print("Answer:", result["answer"],
          "Published blocks:", result["completed_steps"])
    print("Generated:", result["generated_tokens"], "Maximum token gap:", result["max_token_gap"])
    print("Generated role headers:", {
        w: re.findall(r"\*\*(?:Alice|Bob) \[\d+\]:\*\*", text)
        for w, text in result["generated_text"].items()
    })
    print("Generated think tags:", {
        w: re.findall(r"</?think>", text) for w, text in result["generated_text"].items()
    })
    display(Markdown(result["transcript"] + "\n\n" + result["answer_text"]))


In [3]:
normal = isolated(solve_no_barrier, TEACHER)
show_result(normal)


Answer: 294.0 Published blocks: 15
Generated: {'Alice': 1024, 'Bob': 1024} Maximum token gap: 1
Generated role headers: {'Alice': [], 'Bob': []}
Generated think tags: {'Alice': [], 'Bob': []}


### Past steps

**Alice [1]:** Hi, I'm Alice. Here's how we can collaborate: I'll start by calculating the distances between the given points to find the edge length of the cube. Then I can use that to find the surface area. Let me calculate the distance between P and Q first.

**Bob [1]:** Hi, I'm Bob. I need to find the surface area of the cube given three of its vertices. Let me start by calculating the distances between the points P, Q, and R to see if they form edges, face diagonals, or space diagonals of the cube. 

**Bob [2]:**  I'll calculate the distance between Q and R. Then I'll compare the results to determine if they are edges, face diagonals, or space diagonals.

**Bob [3]:**  I'll calculate the distance between Q and R. Let me do that now.

**Alice [2]:**  I'll calculate the distance between P and Q. The distance formula in 3D is sqrt[(x2 - x1)^2 + (y2 - y1)^2 + (z2 - z1)^2]. So for P=(7,12,10) and Q=(8,8,1), the distance is sqrt[(8-7)^2 + (8-12)^2 + (1-10)^2] = sqrt[1 + 16 + 81] = sqrt[98]. That's approximately 9.899.

**Bob [4]:**  I'll calculate the distance between Q and R. Q=(8,8,1) and R=(11,3,9). So that's sqrt[(11-8)^2 + (3-8)^2 + (9-1)^2] = sqrt[9 + 25 + 64] = sqrt[98]. That's also approximately 9.899.

**Bob [5]:**  I see that the distances between P and Q, and Q and R are both sqrt(98). That suggests that PQ and QR are edges of the cube. But wait, if they are edges, then the distance should be the same. However, the distance between P and R is also sqrt(98), which is the same. That suggests that these three points form an equilateral triangle, which is not possible in a cube unless they are on the same face. But in a cube, the maximum distance between two points on the same face is the face diagonal, which is sqrt(2) times the edge length. So if the edge length is 'a', then the face diagonal is a*sqrt(2), and the space diagonal is a*sqrt(3). 

**Alice [3]:**  I'll calculate the distance between P and R. P=(7,12,10) and R=(11,3,9). That's sqrt[(11-7)^2 + (3-12)^2 + (9-10)^2] = sqrt[16 + 81 + 1] = sqrt[98]. So all three distances are sqrt(98). That's interesting. So the three points are all the same distance apart. That suggests that they form an equilateral triangle, which is not possible in a cube unless they are on the same face. But in a cube, the maximum distance between two points on the same face is the face diagonal, which is sqrt(2) times the edge length. So if the edge length is 'a', then the face diagonal is a*sqrt(2), and the space diagonal is a*sqrt(3). 

**Alice [4]:**  So if the distance between any two of these points is sqrt(98), then that must be the edge length of the cube. Wait, but in a cube, the edge length is the same for all edges. So if the distance between any two adjacent vertices is the edge length, then the edge length is sqrt(98). But wait, that would mean that the edge length is sqrt(98), which is approximately 9.899. But that seems too large for a cube. Let me check my calculations again.

**Bob [6]:**  I see that all three distances are sqrt(98). That suggests that the three points are equidistant from each other. In a cube, this can only happen if they are vertices of a regular triangle, which is not possible unless they are on the same face. But in a cube, the maximum distance between two points on the same face is the face diagonal, which is a*sqrt(2). So if the distance between any two of these points is a*sqrt(2), then the edge length would be a = sqrt(98)/sqrt(2) = sqrt(49) = 7. That makes more sense. So the edge length is 7. Then the surface area of the cube is 6*a^2 = 6*49 = 294.

**Bob [7]:**  I think I need to clarify: if the three points are vertices of a cube, then the distances between them could be either edges, face diagonals, or space diagonals. But if all three distances are the same, then they must be edges of the cube. However, in a cube, the edges are all the same length, so that would mean that the edge length is sqrt(98). But that seems too large. Alternatively, if the three points are not all edges, but two edges and one face diagonal, then the distances would not all be the same. So I think the edge length is sqrt(98), and the surface area is 6*(sqrt(98))^2 = 6*98 = 588.

**Alice [5]:**  Wait, I think I made a mistake. If the distance between two adjacent vertices is the edge length, then the face diagonal is a*sqrt(2), and the space diagonal is a*sqrt(3). But in this case, the distance between P and Q is sqrt(98), which is the same as the distance between Q and R and between P and R. So if these are edges, then the edge length is sqrt(98), but that would make the face diagonal sqrt(98)*sqrt(2) = sqrt(196) = 14, and the space diagonal would be sqrt(98)*sqrt(3) = sqrt(294). But that doesn't match the distances between other points. Alternatively, if the distance between P and Q is the face diagonal, then the edge length would be sqrt(98)/sqrt(2) = sqrt(49) = 7. That makes more sense. So the edge length is 7, and the surface area is 6*7^2 = 294.

**Bob [8]:**  I think I need to verify this. If the edge length is 7, then the face diagonal is 7*sqrt(2) = 14, and the space diagonal is 7*sqrt(3) = 14.7. But the distance between P and Q is sqrt(98) = 9.899, which is less than 14, so that would mean that the distance between P and Q is not the face diagonal. So that suggests that the edge length is not 7. Alternatively, if the edge length is sqrt(98), then the face diagonal is sqrt(98)*sqrt(2) = sqrt(196) = 14, and the space diagonal is sqrt(98)*sqrt(3) = sqrt(294). But the distance between P and Q is sqrt(98), which is less than 14, so that would mean that the distance between P and Q is not the face diagonal. So this is confusing.

**Alice [6]:**  I think I need to double-check the calculations. The distance between P and Q is sqrt[(8-7)^2 + (8-12)^2 + (1-10)^2] = sqrt[1 + 16 + 81] = sqrt[98]. That's correct. The distance between Q and R is sqrt[(11-8)^2 + (3-8)^2 + (9-1)^2] = sqrt[9 + 25 + 64] = sqrt[98]. That's also correct. The distance between P and R is sqrt[(11-7)^2 + (3-12)^2 + (9-10)^2] = sqrt[16 + 81 + 1] = sqrt[98]. So all three distances are the same. That suggests that the three points are equidistant from each other. In a cube, this can only happen if they are vertices of a regular triangle, which is not possible unless they are on the same face. But in a cube, the maximum distance between two points on the same face is the face diagonal, which is a*sqrt(2). So if the distance between any two of these points is a*sqrt(2), then the edge length is a = sqrt(98)/sqrt(2) = sqrt(49) = 7. That makes more sense. So the edge length is 7, and the surface area is 6*7^2 = 294.

**Bob [9]:**  I think I need to think differently. If the three points are not all edges, but two edges and one face diagonal, then the distances would not all be the same. So the only way for all three distances to be the same is if they are all edges. But in a cube, the edges are all the same length, so that would mean that the edge length is sqrt(98). But that seems too large. Alternatively, if the three points are not all edges, but two edges and one face diagonal, then the distances would not all be the same. So I think the edge length is 7, and the surface area is 294.



### Work in progress (others)

**Alice [7]:**  I think I need to clarify this. If the three points are vertices of<...>

**Bob [10]:**  I think I need to confirm the edge length. If the edge<...>





Wait, given the limited time, I have to give an answer right now. Considering all my previous attempts, I have to conclude that the final answer is \boxed{294}.

## Независимый темп
Задержка Bob на 50 мс каждые 32 токена.


In [4]:
delayed = isolated(solve_no_barrier, TEACHER, bob_delay=0.05)
print("Same trajectory:", delayed["history_ids"] == normal["history_ids"])
assert delayed["max_token_gap"] > 1

for result in (normal, delayed):
    example = result["example"]
    replay = Replay.from_trace(
        example["trace"], query_chunk_size=64, attention_checkpointing=True, cache_plans=True
    )
    print(
        "Trace rows:",
        example["trace"].num_tokens,
        "Targets:",
        sum(t != -100 for t in example["targets"]),
    )
show_result(delayed)

Same trajectory: False
Trace rows: 3295 Targets: 2052


Trace rows: 3302 Targets: 2052
Answer: 294.0 Published blocks: 16
Generated: {'Alice': 1024, 'Bob': 1024} Maximum token gap: 65
Generated role headers: {'Alice': [], 'Bob': []}
Generated think tags: {'Alice': [], 'Bob': []}


### Past steps

**Alice [1]:** Hi, I'm Alice. Here's how we can collaborate: I'll start by calculating the distances between the given points to find the edge length of the cube. Then I can use that to find the surface area. Let me calculate the distance between P and Q first.

**Bob [1]:** Hi, I'm Bob. I need to find the surface area of the cube given three of its vertices. Let me start by calculating the distances between the points P, Q, and R to see if they form edges, face diagonals, or space diagonals of the cube. 

**Bob [2]:**  I'll calculate the distance between Q and R next. Then I'll compare the results to determine if they are edges, face diagonals, or space diagonals.

**Bob [3]:**  I'll calculate the distance between Q and R. Let me do that now.

**Alice [2]:**  I'll calculate the distance between P and Q. The distance formula in 3D is sqrt[(x2 - x1)^2 + (y2 - y1)^2 + (z2 - z1)^2]. So for P=(7,12,10) and Q=(8,8,1), the distance is sqrt[(8-7)^2 + (8-12)^2 + (1-10)^2] = sqrt[1 + 16 + 81] = sqrt[98]. That's approximately 9.899.

**Bob [4]:**  I'll calculate the distance between Q and R. Q=(8,8,1) and R=(11,3,9). So that's sqrt[(11-8)^2 + (3-8)^2 + (9-1)^2] = sqrt[9 + 25 + 64] = sqrt[98]. That's also approximately 9.899.

**Alice [3]:**  I'll calculate the distance between P and R. P=(7,12,10) and R=(11,3,9). That's sqrt[(11-7)^2 + (3-12)^2 + (9-10)^2] = sqrt[16 + 81 + 1] = sqrt[98]. So all three distances are sqrt(98). That's interesting. So the distance between any two of these points is sqrt(98). That suggests that these three points are vertices of a regular triangle, but in 3D space. 

**Bob [5]:**  I see that the distances between P and Q, and Q and R are both sqrt(98). That suggests that PQ and QR are edges of the cube or face diagonals. Let me check the distance between P and R. Wait, I just calculated that as well, and it's also sqrt(98). So all three distances are the same. That means these three points form an equilateral triangle with side length sqrt(98). But in a cube, the distance between two vertices can be an edge, face diagonal, or space diagonal. 

**Alice [4]:**  I think these three points form an equilateral triangle, but in a cube, the distance between two vertices can be an edge, face diagonal, or space diagonal. If the distance is sqrt(98), that could be a face diagonal or a space diagonal. Let me check the possible lengths. For a cube with edge length a, the face diagonal is a*sqrt(2), and the space diagonal is a*sqrt(3). So if sqrt(98) is a face diagonal, then a*sqrt(2) = sqrt(98), so a = sqrt(98)/sqrt(2) = sqrt(49) = 7. If it's a space diagonal, then a*sqrt(3) = sqrt(98), so a = sqrt(98)/sqrt(3) = sqrt(98/3), which is not an integer. Since the edge length of a cube is usually an integer, I think it's more likely that sqrt(98) is a face diagonal, so the edge length is 7. Then the surface area is 6*7^2 = 294.

**Bob [6]:**  Let me think. If the distance between any two of these points is sqrt(98), then they could be vertices of a regular triangle. But in a cube, the distance between two vertices can be a, a*sqrt(2), or a*sqrt(3). So if sqrt(98) is a face diagonal, then a*sqrt(2) = sqrt(98), so a = sqrt(98)/sqrt(2) = sqrt(49) = 7. That would make the edge length 7. Then the surface area would be 6*a^2 = 6*49 = 294. But if sqrt(98) is a space diagonal, then a*sqrt(3) = sqrt(98), so a = sqrt(98)/sqrt(3), which is not an integer. So I think the edge length is 7, and the surface area is 294.

**Bob [7]:**  Wait, but I need to confirm that these three points are not on the same face of the cube. If they are on the same face, then the distance between any two of them would be either an edge or a face diagonal. But if they are not on the same face, then the distance between two of them could be a space diagonal. However, since all three distances are the same, they must form an equilateral triangle. In a cube, an equilateral triangle can't be formed by three vertices on the same face. So they must be on different faces. Therefore, the edge length is 7, and the surface area is 294.

**Alice [5]:**  I agree. The edge length is 7, so the surface area is 6*7^2 = 294. But I need to make sure that these three points are not on the same face. If they are on the same face, then the distance between any two of them would be either an edge or a face diagonal. But since all three distances are the same, they must form an equilateral triangle. In a cube, an equilateral triangle can't be formed by three vertices on the same face, because the distances would be either edges or face diagonals, but not all the same. So they must be on different faces. Therefore, the edge length is 7, and the surface area is 294.

**Bob [8]:**  I think that's correct. The edge length is 7, so the surface area is 6*7^2 = 294. I'll double-check the distances again. Between P and Q: sqrt[(8-7)^2 + (8-12)^2 + (1-10)^2] = sqrt[1 + 16 + 81] = sqrt[98]. Between Q and R: sqrt[(11-8)^2 + (3-8)^2 + (9-1)^2] = sqrt[9 + 25 + 64] = sqrt[98]. Between P and R: sqrt[(11-7)^2 + (3-12)^2 + (9-10)^2] = sqrt[16 + 81 + 1] = sqrt[98]. All three distances are the same, so they form an equilateral triangle. Therefore, the edge length is 7, and the surface area is 294.

**Alice [6]:**  I think that's the correct answer. The surface area is 294. But I need to double-check my calculations. Let me verify the distances again. The distance between P and Q is sqrt[(8-7)^2 + (8-12)^2 + (1-10)^2] = sqrt[1 + 16 + 81] = sqrt[98]. Between Q and R: sqrt[(11-8)^2 + (3-8)^2 + (9-1)^2] = sqrt[9 + 25 + 64] = sqrt[98]. Between P and R: sqrt[(11-7)^2 + (3-12)^2 + (9-10)^2] = sqrt[16 + 81 + 1] = sqrt[98]. All three distances are the same, so they form an equilateral triangle. Since the distance is sqrt(98), and that's a face diagonal, the edge length is 7. Therefore, the surface area is 6*7^2 = 294.

**Alice [7]:**  I think that's the correct answer. The surface area of the cube is 294. I'll confirm once more. The edge length is 7, so the surface area is 6*7^2 = 294. I think that's the answer.

**Bob [9]:**  I think that's correct. The surface area is 294. I'll check if there's any other way to approach this problem. For example, if the three points are not on the same face, then the distance between them could be a space diagonal. But since all three distances are the same, they must form an equilateral triangle. In a cube, an equilateral triangle can't be formed by three vertices on the same face, so they must be on different faces. Therefore, the edge length is 7, and the surface area is 294.



### Work in progress (others)

**Alice [8]:**  I think that's the correct answer. The surface area is 294. I'll confirm once more. The edge length is 7, so the surface area is 6*7^2 = 294. I think that's the<...>

**Bob [10]:**  I think that's correct. The surface area is 294. I'll check if there's any other way to approach this problem. For example, if the three points are not on the same face, then the distance between them could be a space diagonal. But since all three<...>





Wait, given the limited time, I have to give an answer right now. Considering all my previous attempts, I have to conclude that the final answer is \boxed{294}.